# Model Serving Platform: FastAPI + Docker + Monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/aiml-companion/blob/main/projects/model-serving-platform/notebooks/Model_Serving_Platform.ipynb)

> Build production ML infrastructure: model server, health checks, metrics, and load testing.

**Note:** Full Docker/CI deployment runs locally. This notebook demos core components.

## Setup

In [ ]:
!pip install scikit-learn joblib numpy -q

## Step 1: Train & Save Model

In [ ]:
import numpy as np, joblib, os, time
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"Accuracy: {model.score(X_test, y_test):.4f}")

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/model.joblib")
print("Model saved")

## Step 2: Model Server (FastAPI Components)

In [ ]:
model_server = joblib.load("models/model.joblib")
model_version = "v1.0.0"
start_time = time.time()

def predict(features):
    arr = np.array(features).reshape(1, -1)
    pred = int(model_server.predict(arr)[0])
    prob = float(model_server.predict_proba(arr).max())
    return {"prediction": pred, "probability": prob, "model_version": model_version}

def health():
    return {"status": "healthy", "model_loaded": True, "version": model_version,
            "uptime_s": round(time.time() - start_time, 1)}

print("Health:", health())
print("Predict:", predict([5.1, 3.5, 1.4, 0.2]))

## Step 3: Metrics Collection

In [ ]:
from collections import defaultdict

class Metrics:
    def __init__(self):
        self.latencies = []
        self.predictions = defaultdict(int)
        self.errors = defaultdict(int)
    def record(self, latency, pred_class, version):
        self.latencies.append(latency)
        self.predictions[f"{version}:{pred_class}"] += 1
    def report(self):
        lats = sorted(self.latencies)
        return {"total": len(lats),
                "p50_ms": round(np.percentile(lats, 50)*1000, 2),
                "p95_ms": round(np.percentile(lats, 95)*1000, 2),
                "p99_ms": round(np.percentile(lats, 99)*1000, 2)}

metrics = Metrics()
for _ in range(100):
    features = [np.random.normal(loc, 0.5) for loc in [5.8, 3.0, 3.7, 1.2]]
    t0 = time.time()
    r = predict(features)
    metrics.record(time.time() - t0, r["prediction"], r["model_version"])

print("Metrics:", metrics.report())

## Step 4: Load Test Simulation

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import random

def sim_request():
    if random.random() < 0.1:
        t0 = time.time()
        health()
        return ("health", time.time() - t0)
    features = [random.gauss(0, 1) for _ in range(4)]
    t0 = time.time()
    predict(features)
    return ("predict", time.time() - t0)

print("Running 500 requests across 10 threads...")
with ThreadPoolExecutor(max_workers=10) as ex:
    results = list(ex.map(lambda _: sim_request(), range(500)))

pred_lats = sorted([lat for typ, lat in results if typ == "predict"])
print(f"Predictions: {len(pred_lats)}")
print(f"P50: {np.percentile(pred_lats, 50)*1000:.1f}ms")
print(f"P95: {np.percentile(pred_lats, 95)*1000:.1f}ms")
print(f"P99: {np.percentile(pred_lats, 99)*1000:.1f}ms")
print(f"Throughput: {len(pred_lats)/sum(pred_lats):.0f} req/s")

## Docker & CI/CD (Run Locally)

```bash
git clone https://github.com/genieincodebottle/aiml-companion.git
cd aiml-companion/projects/model-serving-platform
docker build -t ml-server -f docker/Dockerfile .
docker run -p 8000:8000 ml-server
# Then: curl localhost:8000/health
```

The repo includes Dockerfile, docker-compose.yml, CI/CD workflow, and Locust load tests.

## Key Takeaways

- **Lifespan loading**: Load model once at startup, not per-request
- **Health checks**: Docker HEALTHCHECK + /health endpoint
- **Graceful shutdown**: SIGTERM handler for in-flight requests
- **P50/P95/P99**: Track tail latency, not just averages
- **Load testing**: 90/10 predict/health split simulates real traffic